In [ ]:
import logging
import os
import pickle
import random
import sys
from os.path import join

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import xgboost as xgb
from matplotlib.font_manager import FontProperties, fontManager
from sklearn.metrics import matthews_corrcoef, roc_auc_score
from sklearn.model_selection import KFold

sys.path.append("./additional_code")
# from data_preprocessing import *
CURRENT_DIR = os.getcwd()
print(CURRENT_DIR)
our_data = CURRENT_DIR + "/../data/our_data/"

In [ ]:
queryname = "Arabidopsis_thaliana"
# queryname = "Erigeron_breviscapus"
queryname = "Glycine_max"
# queryname = "Zea_mays"

p450_name	complex_name	p450_score	affinity	cnn_score	cnn_affi	wei_score	dock_num	if_right

In [ ]:
result = pd.read_pickle(our_data + f"screening2/bin/{queryname}_bu.pkl")

In [ ]:
alldatagroup = result.groupby("enzyme")
indexs = []
for index, group in alldatagroup:
    indexs.append(len(group))
    if index == "CYP72A69":
        break
        if len(group) != result["substrate"].nunique():
            print(index)

In [ ]:
group

In [ ]:
filtered_rows = result[(result["if_right"] == True) & (result["Binding"] == 1)]

In [ ]:
filtered_rows

In [ ]:
font_path = "data/font/TimesNewRoman.ttf"
font_prop = FontProperties(fname=font_path, size=14)

fontManager.addfont(font_path)


font_name = font_prop.get_name()

plt.rcParams["font.family"] = font_name
plt.rcParams["font.size"] = 14

In [ ]:
# from matplotlib.colors import LinearSegmentedColormap
import matplotlib.cm as cm

enzymes = filtered_rows["enzyme"].unique().tolist()
substrates = [
    filtered_rows[(filtered_rows["enzyme"] == i) & (filtered_rows["Binding"] == 1)][
        "substrate"
    ].item()
    for i in enzymes
]

mywidth = 0.6
myfigwidth = len(enzymes) * (2 + 0.5)
myfighigh = 14

data_to_plot = []
for num, nowsubstrate in enumerate(substrates):
    resultcc = result[
        (
            result["cc"]
            == (
                result[
                    (result["enzyme"] == enzymes[num]) & (result["if_right"] == True)
                ]["cc"].item()
            )
        )
        | (result["cc"].isnull())
    ]
    data_to_plot.append(
        (100 - 70) / 100 * (resultcc[resultcc["substrate"] == nowsubstrate]["scores"])
        + (70) / 100 * resultcc[resultcc["substrate"] == nowsubstrate]["wei_score"]
    )
# ------

plt.figure(dpi=300)
plt.figure(figsize=(myfigwidth, myfighigh))
cmap = cm.Set3
plt.subplots_adjust(top=0.9, bottom=0.3)
plt.ylim(0, 1)
plt.tick_params(axis="y", labelsize=30)
box = plt.boxplot(
    data_to_plot,
    positions=range(1, len(enzymes) + 1),
    widths=mywidth,
    patch_artist=True,
)


for i, patch in enumerate(box["boxes"]):
    patch.set_facecolor(cmap((i + 1) / (len(enzymes) + 1)))
    patch.set_alpha(0.9)

for median in box["medians"]:
    median.set_color("black")

plt.xticks(range(1, len(enzymes) + 1), enzymes, rotation=30, fontsize=40)
plt.ylabel("Scores", fontsize=50)

for i, nowenzyme in enumerate(enzymes, start=1):
    substrate_scores = ((100 - 70) / 100) * filtered_rows[
        (filtered_rows["enzyme"] == nowenzyme) & (filtered_rows["enzyme"] == nowenzyme)
    ]["scores"] + (70) / 100 * filtered_rows[
        (filtered_rows["enzyme"] == nowenzyme) & (filtered_rows["enzyme"] == nowenzyme)
    ]["wei_score"]
    # all_scores_cache = ((100-70)/100)*(result[result['substrate'] == substrates[i-1]]['scores'])+(70)/100*(result[result['substrate'] == substrates[i-1]]['wei_score'])
    # ----new------
    resultccpoint = result[
        (
            result["cc"]
            == (
                result[(result["enzyme"] == nowenzyme) & (result["if_right"] == True)][
                    "cc"
                ].item()
            )
        )
        | (result["cc"].isnull())
    ]
    all_scores_cache = (100 - 70) / 100 * (
        resultccpoint[resultccpoint["substrate"] == substrates[i - 1]]["scores"]
    ) + (70) / 100 * resultccpoint[resultccpoint["substrate"] == substrates[i - 1]][
        "wei_score"
    ]
    # ------------
    all_scores_cache.sort_values(ascending=False, inplace=True)
    all_scores_cache.reset_index(drop=True, inplace=True)
    rank_cache = all_scores_cache.tolist().index(substrate_scores.item()) + 1
    # ranking= f'{rank_cache}/{len(all_scores_cache)}'
    ranking = str(round(rank_cache / len(all_scores_cache) * 100, 1)) + "%"
    x_coords = [i] * len(substrate_scores)
    plt.scatter(x_coords, substrate_scores, color="red", alpha=0.7, zorder=3, s=200)
    plt.annotate(
        ranking,
        (i, substrate_scores),
        textcoords="offset points",
        xytext=(2, 24),
        ha="center",
        fontsize=34,
    )


# plt.tight_layout()
plt.savefig(f"cache/{queryname}_boxplot.png")
plt.savefig(f"cache/{queryname}_boxplot.svg")
plt.show()

In [ ]:
dfoutcsv = result[
    [
        "enzyme",
        "substrate",
        "Binding",
        "if_right",
        "scores",
        "wei_score",
        "cc",
    ]
]
dfoutcsv = dfoutcsv.rename(
    columns={
        "wei_score": "Gnina_score",
        "scores": "ESP_p450_score",
    }
)
dfoutcsv.to_csv(f"cache/{queryname}.csv", index=False)

In [ ]:
dfoutcsv["enzyme"].nunique()

In [ ]:
len(substrates)